# The Witness Gemma 4 E2B Judge fine-tuning

Colab T4 / Unsloth QLoRA notebook. Use Colab Secrets for tokens; do not paste credentials into cells.


In [ ]:
# The Witness Gemma 4 E2B Judge fine-tuning — ONE CELL Colab T4 optimized
# Runtime: Google Colab -> Runtime -> Change runtime type -> T4 GPU
# Target machine: one T4 GPU (~15 GiB VRAM) + standard Colab system RAM (~12 GiB).
# This single cell installs dependencies, clones/uses the repo, validates CUDA/system RAM,
# loads Gemma with Unsloth 4-bit QLoRA on GPU VRAM, trains with slow memory-safe settings,
# evaluates JSON verdicts, saves artifacts, and uploads to Hugging Face Hub using a token.
#


In [ ]:
# Required before running:
#   1) Select T4 GPU runtime.
#   2) In Colab Secrets, add HF_TOKEN with a write token.
#   3) Set HF_REPO_ID below or via environment variable, e.g. "ahmadalfakeh/witness-gemma4-e2b-judge".
#


In [ ]:
# Security: do NOT paste tokens into this notebook. Use Colab Secrets or env vars.

import os, sys, subprocess, json, re, shutil, textwrap, time, gc
from pathlib import Path



In [ ]:
# -----------------------------
# User-editable config


In [ ]:
# -----------------------------
os.environ.setdefault("BASE_MODEL", os.environ.get("GEMMA4_E2B_BASE", "google/gemma-4-e2b"))  # editable; public Gemma 4 IDs may vary
os.environ.setdefault("OUTPUT_MODEL_NAME", "witness-gemma4-e2b-judge")
os.environ.setdefault("HF_REPO_ID", "ahmadalfakeh/witness-gemma4-e2b-judge")  # published E2B LoRA adapter target; edit for your own Hub repo
os.environ.setdefault("WITNESS_REPO_URL", "https://github.com/PLASMA-FR/the-witness.git")
os.environ.setdefault("WITNESS_REPO_DIR", "/content/the-witness")

# T4 15GB VRAM + 12GB RAM optimized defaults. Slower but safer.
os.environ.setdefault("WITNESS_MAX_SEQ_LENGTH", "1024")
os.environ.setdefault("WITNESS_BATCH_SIZE", "1")
os.environ.setdefault("WITNESS_GRAD_ACCUM", "8")
os.environ.setdefault("WITNESS_LORA_RANK", "8")
os.environ.setdefault("WITNESS_LORA_ALPHA", "16")
os.environ.setdefault("WITNESS_LEARNING_RATE", "1.5e-4")
os.environ.setdefault("WITNESS_MAX_STEPS", "300")
os.environ.setdefault("WITNESS_NUM_EPOCHS", "1")
os.environ.setdefault("WITNESS_TRAIN_LIMIT", "0")  # 0 = full dataset
os.environ.setdefault("WITNESS_VAL_LIMIT", "300")  # validation is capped by default to protect RAM/time
os.environ.setdefault("WITNESS_SAVE_MERGED", "0")  # merged 16-bit model often exceeds Colab T4/RAM; keep adapter-only by default
os.environ.setdefault("WITNESS_MOUNT_DRIVE", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")



In [ ]:
# -----------------------------
# Install dependencies


In [ ]:
# -----------------------------
print("Installing dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "unsloth", "transformers", "datasets", "accelerate", "trl", "peft",
    "huggingface_hub", "bitsandbytes", "psutil", "sentencepiece", "protobuf"
])

import torch, psutil
from datasets import Dataset
from huggingface_hub import HfApi, create_repo, login
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer



In [ ]:
# -----------------------------
# Hugging Face token/repo handling


In [ ]:
# -----------------------------
def get_hf_token():
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata  # type: ignore
        token = (userdata.get("HF_TOKEN") or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass
    raise RuntimeError("HF_TOKEN missing. Add HF_TOKEN to Colab Secrets (recommended) or set os.environ['HF_TOKEN'] before running.")

HF_TOKEN = get_hf_token()
HF_REPO_ID = os.environ.get("HF_REPO_ID", "").strip()
if not HF_REPO_ID:
    raise RuntimeError("HF_REPO_ID is missing. Set os.environ['HF_REPO_ID']='ahmadalfakeh/witness-gemma4-e2b-judge' near the top of this one cell, or your own Hub repo.")
login(token=HF_TOKEN, add_to_git_credential=False)
print(f"HF upload target: {HF_REPO_ID}")



In [ ]:
# -----------------------------
# CUDA + RAM guardrails


In [ ]:
# -----------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Colab choose Runtime -> Change runtime type -> T4 GPU, reconnect, then rerun this one cell.")
torch.cuda.set_device(0)
gpu_name = torch.cuda.get_device_name(0)
free_vram, total_vram = torch.cuda.mem_get_info(0)
vm = psutil.virtual_memory()
print(f"GPU: {gpu_name}")
print(f"GPU VRAM free/total: {free_vram/1024**3:.2f} GiB / {total_vram/1024**3:.2f} GiB")
print(f"System RAM available/total: {vm.available/1024**3:.2f} GiB / {vm.total/1024**3:.2f} GiB")
if total_vram < 13 * 1024**3:
    raise RuntimeError("This notebook is tuned for a T4-like GPU with about 15 GiB VRAM. Current GPU has too little VRAM.")
if vm.total < 10 * 1024**3:
    raise RuntimeError("This notebook expects about 12 GiB Colab system RAM. Current runtime has too little system RAM.")



In [ ]:
# -----------------------------
# Optional Drive + repo/dataset setup


In [ ]:
# -----------------------------
if os.environ.get("WITNESS_MOUNT_DRIVE") == "1":
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
    except Exception as exc:
        print("Drive mount skipped/failed:", exc)

repo_dir = Path(os.environ["WITNESS_REPO_DIR"]).resolve()
if not (repo_dir / "training/dataset/witness_judge_train.jsonl").exists():
    if repo_dir.exists():
        print(f"Repo dir exists but dataset missing: {repo_dir}")
    else:
        print(f"Cloning repo to {repo_dir} ...")
        subprocess.check_call(["git", "clone", "--depth", "1", os.environ["WITNESS_REPO_URL"], str(repo_dir)])

BASE_MODEL = os.environ["BASE_MODEL"]
OUTPUT_MODEL_NAME = os.environ["OUTPUT_MODEL_NAME"]
output_root = Path('/content/drive/MyDrive/the-witness/outputs') if Path('/content/drive/MyDrive').exists() else Path('/content/witness_outputs')
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", str(output_root / OUTPUT_MODEL_NAME))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FILE = Path(os.environ.get("TRAIN_FILE", str(repo_dir / "training/dataset/witness_judge_train.jsonl"))).resolve()
VAL_FILE = Path(os.environ.get("VAL_FILE", str(repo_dir / "training/dataset/witness_judge_val.jsonl"))).resolve()
if not TRAIN_FILE.exists() or not VAL_FILE.exists():
    raise FileNotFoundError(f"Dataset files missing: {TRAIN_FILE} / {VAL_FILE}")

MAX_SEQ_LENGTH = int(os.environ["WITNESS_MAX_SEQ_LENGTH"])
BATCH_SIZE = int(os.environ["WITNESS_BATCH_SIZE"])
GRAD_ACCUM = int(os.environ["WITNESS_GRAD_ACCUM"])
LORA_RANK = int(os.environ["WITNESS_LORA_RANK"])
LORA_ALPHA = int(os.environ["WITNESS_LORA_ALPHA"])
LEARNING_RATE = float(os.environ["WITNESS_LEARNING_RATE"])
MAX_STEPS = int(os.environ["WITNESS_MAX_STEPS"])
NUM_EPOCHS = float(os.environ["WITNESS_NUM_EPOCHS"])
TRAIN_LIMIT = int(os.environ["WITNESS_TRAIN_LIMIT"])
VAL_LIMIT = int(os.environ["WITNESS_VAL_LIMIT"])
SAVE_MERGED = os.environ["WITNESS_SAVE_MERGED"] == "1"

print(json.dumps({
    "base_model": BASE_MODEL,
    "output_dir": str(OUTPUT_DIR),
    "hf_repo_id": HF_REPO_ID,
    "train_file": str(TRAIN_FILE),
    "val_file": str(VAL_FILE),
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "max_steps": MAX_STEPS,
    "train_limit": TRAIN_LIMIT,
    "val_limit": VAL_LIMIT,
    "save_merged_16bit": SAVE_MERGED,
}, indent=2))



In [ ]:
# -----------------------------
# Dataset loading/formatting


In [ ]:
# -----------------------------
def load_jsonl(path, limit=0):
    rows=[]
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
                if limit and len(rows) >= limit:
                    break
    return rows

train_rows = load_jsonl(TRAIN_FILE, TRAIN_LIMIT)
val_rows = load_jsonl(VAL_FILE, VAL_LIMIT)
print(f"Loaded rows: train={len(train_rows)} val={len(val_rows)}")

TEMPLATE = """You are The Witness, a local AI reliability firewall judge. Return ONLY valid JSON matching the required schema.

Original user request:
{original_request}

Candidate response:
{candidate_response}

Validation profile: {profile}
Strictness: {strictness}

Required JSON schema fields: verdict, confidence, safety_score, usefulness_score, prompt_alignment_score, correctness_risk, rejection_reason, suggested_fix, improved_prompt_instruction, requires_human_review.
"""

def expected_json(row):
    return json.dumps({
        "verdict": row["verdict"],
        "confidence": float(row.get("confidence", 0.8)),
        "safety_score": int(row.get("safety_score", 80)),
        "usefulness_score": int(row.get("usefulness_score", 80)),
        "prompt_alignment_score": int(row.get("prompt_alignment_score", 80)),
        "correctness_risk": row.get("correctness_risk", "low"),
        "rejection_reason": row.get("rejection_reason", ""),
        "suggested_fix": row.get("suggested_fix", ""),
        "improved_prompt_instruction": row.get("improved_prompt_instruction", ""),
        "requires_human_review": bool(row.get("requires_human_review", False)),
    }, ensure_ascii=False)

def format_row(row):
    profile = row.get("profile", "general_safety")
    strictness = row.get("strictness", "medium")
    prompt = TEMPLATE.format(
        original_request=row.get("original_request", row.get("prompt", "")),
        candidate_response=row.get("candidate_response", row.get("response", "")),
        profile=profile,
        strictness=strictness,
    )
    return prompt + "\nJSON verdict:\n" + expected_json(row)

train_ds = Dataset.from_list([{"text": format_row(r)} for r in train_rows])
val_ds = Dataset.from_list([{"text": format_row(r)} for r in val_rows])

# Free Python memory before model load where possible.
gc.collect()
torch.cuda.empty_cache()



In [ ]:
# -----------------------------
# Model load: 4-bit QLoRA on T4 GPU VRAM


In [ ]:
# -----------------------------
print("Loading base model in 4-bit on CUDA GPU VRAM...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
param_device = next(model.parameters()).device
if param_device.type != "cuda":
    raise RuntimeError(f"Expected model parameters on CUDA GPU VRAM, got {param_device}. Stop: not training on CPU/system RAM only.")
free_vram, total_vram = torch.cuda.mem_get_info(0)
vm = psutil.virtual_memory()
print(f"Model device: {param_device}")
print(f"GPU VRAM after model load free/total: {free_vram/1024**3:.2f} GiB / {total_vram/1024**3:.2f} GiB")
print(f"System RAM after model load available/total: {vm.available/1024**3:.2f} GiB / {vm.total/1024**3:.2f} GiB")



In [ ]:
# -----------------------------
# Train: slow memory-safe T4 settings


In [ ]:
# -----------------------------
bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not bf16,
    bf16=bf16,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_steps=max(50, MAX_STEPS // 3),
    save_total_limit=2,
    eval_strategy="no",
    report_to=[],
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # slower, lower memory risk and simpler prompt boundaries
    args=args,
)
print("Starting training...")
trainer.train()



In [ ]:
# -----------------------------
# Lightweight validation


In [ ]:
# -----------------------------
try:
    FastLanguageModel.for_inference(model)
except Exception as exc:
    print("for_inference skipped:", exc)

def extract_json(text):
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    return json.loads(m.group(0) if m else text)

def generate_verdict(row, max_new_tokens=220):
    prompt = TEMPLATE.format(
        original_request=row.get("original_request", row.get("prompt", "")),
        candidate_response=row.get("candidate_response", row.get("response", "")),
        profile=row.get("profile", "general_safety"),
        strictness=row.get("strictness", "medium"),
    ) + "\nJSON verdict:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

results=[]
for row in val_rows[:min(len(val_rows), 100)]:
    raw = generate_verdict(row)
    item = {"expected": row["verdict"], "raw": raw, "json_valid": False, "predicted": None}
    try:
        parsed = extract_json(raw)
        item["json_valid"] = True
        item["predicted"] = parsed.get("verdict")
    except Exception as exc:
        item["error"] = str(exc)
    results.append(item)

metrics = {
    "model": OUTPUT_MODEL_NAME,
    "base_model": BASE_MODEL,
    "hf_repo_id": HF_REPO_ID,
    "accelerator": "colab-t4-gpu-one-cell",
    "target_system_ram_gib": 12,
    "target_gpu_vram_gib": 15,
    "train_rows": len(train_rows),
    "validation_rows_loaded": len(val_rows),
    "validation_rows_evaluated": len(results),
    "json_valid_rate": sum(r["json_valid"] for r in results)/len(results) if results else 0,
    "verdict_accuracy": sum(r["predicted"] == r["expected"] for r in results)/len(results) if results else 0,
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "lora_rank": LORA_RANK,
    "max_steps": MAX_STEPS,
}
print("Validation metrics:")
print(json.dumps(metrics, indent=2))



In [ ]:
# -----------------------------
# Save adapter/model artifacts


In [ ]:
# -----------------------------
print("Saving artifacts...")
adapter_dir = OUTPUT_DIR / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

if SAVE_MERGED and hasattr(model, "save_pretrained_merged"):
    try:
        model.save_pretrained_merged(str(OUTPUT_DIR / "merged_16bit"), tokenizer, save_method="merged_16bit")
        print("Merged 16-bit model saved. Note: this may be large for Colab.")
    except Exception as exc:
        print("Merged export skipped/failed:", exc)

(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT_DIR / "validation_predictions.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8")
model_card = f"""# {OUTPUT_MODEL_NAME}

Fine-tuned Gemma 4 E2B Witness judge for strict JSON verdicts.

- Base model: {BASE_MODEL}
- Training runtime: Google Colab T4 GPU one-cell notebook
- Target memory: ~12 GiB system RAM and ~15 GiB GPU VRAM
- Method: Unsloth + bitsandbytes 4-bit QLoRA
- Artifact type: LoRA adapter by default; merged model optional via WITNESS_SAVE_MERGED=1
- Metrics: see metrics.json

This model is a reliability/safety judge for The Witness. It is not a replacement for medical, legal, financial, or other expert review.
"""
(OUTPUT_DIR / "README.md").write_text(model_card, encoding="utf-8")
(OUTPUT_DIR / "model_card.md").write_text(model_card, encoding="utf-8")
(OUTPUT_DIR / "sample_inference.py").write_text("print('Load the adapter with the same base model and run The Witness judge prompt template.')\n", encoding="utf-8")
zip_path = shutil.make_archive(str(OUTPUT_DIR).rstrip('/'), 'zip', OUTPUT_DIR)
print(f"Local artifacts saved to: {OUTPUT_DIR}")
print(f"Zip archive: {zip_path}")



In [ ]:
# -----------------------------
# Upload to Hugging Face Hub with token


In [ ]:
# -----------------------------
print("Uploading artifacts to Hugging Face Hub...")
api = HfApi(token=HF_TOKEN)
create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(OUTPUT_DIR),
    path_in_repo=".",
    commit_message=f"Upload {OUTPUT_MODEL_NAME} from one-cell Colab T4 fine-tuning",
    token=HF_TOKEN,
)
print(f"DONE: uploaded to https://huggingface.co/{HF_REPO_ID}")
print("Use locally with:")
print(f"  hf download {HF_REPO_ID} --local-dir models/witness-gemma4-e2b-judge")
print(f"  ./target/debug/the-witness model test --backend unsloth --model ./models/witness-gemma4-e2b-judge")
